## Почему используется локальная LLM для разметки?

### Проблема
Для обучения модели мульти-лейбл классификации нужен размеченный датасет: для каждого фильма - набор тегов (жанр, настроение, тематика, персонажи, сеттинг).

### Почему не вручную?
Разметка 797 фильмов вручную заняла бы десятки часов и была бы субъективной (разные люди ставят разные теги). Это нерационально.

### Почему не через внешние API (OpenAI, GigaChat)?
- **Стоимость**: 797 запросов к платным API - это деньги
- **Приватность**: данные уходят на сторонние серверы
- **Зависимость**: нужен интернет и оплата
- **Лимиты**: бесплатные тарифы ограничивают количество запросов

### Почему локальная LLM?
- **Бесплатно**: модель работает на ПК, локально
- **Без ограничений**: можно обработать все 797 фильмов
- **Приватность**: данные не покидают локальную среду
- **Воспроизводимость**: любой может запустить проект без API-ключей

### Компромисс
Использовать большую LLM для разметки: это медленно, но качественно. А затем обучить маленькую модель, которая будет работать быстро и не требовать больших ресурсов.

In [29]:
import json
import time

import pandas as pd
import requests
from openai import OpenAI
from tqdm import tqdm

Загружаем датафрейм из CSV файла данных из БД

In [30]:
df = pd.read_csv("../data/processed/data_movies.csv")

df.head()

,id,movie_name,year,rating,runtime,meta_score,age_rating,overview,genres,directors,countries
0,1,1+1,2011,8.8,112 минут,57,18.0,Аристократ на коляске нанимает в сиделки бывше...,"Драма, Комедия","Оливье Накаш, Эрик Толедано",Франция
1,2,Интерстеллар,2014,8.7,169 минут,74,18.0,"Когда засуха, пыльные бури и вымирание растени...","Драма, Приключения, Фантастика",Кристофер Нолан,"Великобритания, США"
2,3,Побег из Шоушенка,1994,9.1,142 минуты,82,16.0,Бухгалтер Энди Дюфрейн обвинён в убийстве собс...,Драма,Фрэнк Дарабонт,США
3,4,Остров проклятых,2009,8.5,138 минут,63,18.0,Два американских судебных пристава отправляютс...,"Детектив, Драма, Триллер",Мартин Скорсезе,США
4,5,Джентльмены,2019,8.6,113 минут,51,18.0,Один ушлый американец ещё со студенческих лет ...,"Боевик, Комедия, Криминал",Гай Ричи,"Великобритания, США"


Будет использоваться локальная модель

In [31]:
client = OpenAI(api_key="local", base_url="http://localhost:1234/v1")

## Как устроен промпт для генерации тегов

### Структура промпта:
1. Роль
2. Задача
3. Требования к тегам (не дает модели генерировать длинные фразы)
4. Пример (без него модель может писать целые абзацы)
5. Входные данные
6. Формат ответа (позволяет легко распарсить ответ (`split(',')`))

In [32]:
def create_prompt(movie_name, overview, year, genres, directors, countries):
    """Создает промпт для генерации тегов фильма"""
    prompt = f"""
    
        Ты - эксперт по кино. Твоя задача - сгенерировать теги для фильма на основе его описания.

        Теги должны быть:
        - конкретными и информативными
        - описывать: жанр, настроение, тематику, персонажей, сеттинг, ключевые элементы сюжета
        - на русском языке
        - состоять не более чем из 4 слов

        Пример:
        Фильм: Терминатор
        Описание: Киборг из будущего отправлен в прошлое, чтобы убить Сару Коннор, мать будущего лидера сопротивления.
        Теги: киборг, путешествие во времени, постапокалипсис, боевик, фантастика, культовый, 80-е

        Теперь сгенерируй 5-10 тегов для фильма:
        Фильм: {movie_name}
        Описание: {overview}
        Год выпуска: {year}
        Жанры: {genres}
        Режиссеры: {directors}
        Страны производства: {countries}

        Ответь ТОЛЬКО списком тегов через запятую. Без лишнего текста, без нумерации, без точек в конце.

        Теги:
    """

    return prompt

Функция отправки одного запроса к LLM и получения тегов

In [33]:
def get_tags_llm(movie_name, overview, year, genres, directors, countries):
    """Отправка запроса к LLM и получение тегов"""
    prompt = create_prompt(movie_name, overview, year, genres, directors, countries)

    try:
        response = client.chat.completions.create(
            model="local-model",
            messages=[
                {
                    "role": "system",
                    "content": "Ты - эксперт по кино, генерируешь теги.",
                },
                {"role": "user", "content": prompt},
            ],
            temperature=0.3,
            max_tokens=200,
        )
        tags_text = response.choices[0].message.content

        tags_text = tags_text.replace("Теги:", "").strip()

        tags = [t.strip() for t in tags_text.split(",") if t.strip()]

        return tags

    except requests.exceptions.RequestException as e:
        print(f"Ошибка запроса: {e}")
        return []
    except KeyError as e:
        print(f"Ошибка парсинга ответа: {e}")
        print(
            f"Ответ: {response.text[:200] if 'response' in locals() else 'Нет ответа'}"
        )
        return []

## Зачем нужна функция `get_tags_with_retry`?

LLM является нестабильным инструментом. Она может вернуть пустой ответ, сгенерировать невалидный JSON или упасть с ошибкой.

При ошибке мы делаем до 3 попыток, увеличивая паузу.

In [34]:
def get_tags_with_retry(
    movie_name, overview, year, genres, directors, countries, max_retries=3
):
    """Попытка получить теги, при ошибке повторяет"""
    for attempt in range(max_retries):
        try:
            tags = get_tags_llm(
                movie_name, overview, year, genres, directors, countries
            )
            if tags:
                return tags
        except Exception as e:
            print(f"Попытка {attempt + 1} не удалась для '{movie_name}': {e}")
            time.sleep(2**attempt)
    return []

Для каждого фильма генерируем теги с помощью LLM

In [ ]:
print(f"Всего фильмов для разметки: {len(df)}")

tags_data = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    tags = get_tags_with_retry(
        movie_name=row["movie_name"],
        overview=row["overview"],
        year=row["year"],
        genres=row["genres"],
        directors=row["directors"],
        countries=row["countries"],
    )

    tags_data.append(
        {
            "movie_name": row["movie_name"],
            "id": row["id"],
            "tags": tags,
            "year": row["year"],
            "genres": row["genres"],
            "directors": row["directors"],
            "countries": row["countries"],
        }
    )

    print(f"{tags_data[-1]['movie_name']} - {tags_data[-1]['tags']}")

with open("../data/processed/tags_all.json", "w", encoding="utf-8") as f:
    json.dump(tags_data, f, ensure_ascii=False, indent=2)

print(f"Размечено {len(tags_data)} фильмов")

Проверяем полученный файл с тегами

In [39]:
with open("../data/processed/tags_all.json", "w", encoding="utf-8") as f:
    json.dump(tags_data, f, ensure_ascii=False, indent=2)

i = 0
print(f"Размечено {len(tags_data)} фильмов")
print(f"Вывод для фильма {i}")
print(tags_data[i]['movie_name'])
print(tags_data[i]['tags'])

Размечено 797 фильмов
Вывод для фильма 0
1+1
['аристократ на коляске', 'бывший заключенный', 'искрометная комедия', 'драматичный сюжет', 'французский юмор', 'социальная тема', 'взаимоотношения', '2011 год', 'омар си', 'накаш толедано']
